# Gold_genre_performance

## Import and Load

In [2]:
import pandas as pd

from ecommerce_ingestion.config.constants import DB_OUTPUT_GOLD

In [ ]:
gold_genre_primary_performance = pd.read_parquet(
    DB_OUTPUT_GOLD / "gold_game_catalog.parquet")
gold_genre_secondary_performance = gold_genre_primary_performance
gold_genre_tertiary_performance = gold_genre_primary_performance

gold_genre_primary_performance = gold_genre_primary_performance[
    gold_genre_primary_performance["primary_genre"] != "unknown"]
gold_genre_secondary_performance = gold_genre_secondary_performance[
    gold_genre_secondary_performance["secondary_genre"] != "unknown"]
gold_genre_tertiary_performance = gold_genre_tertiary_performance[
    gold_genre_tertiary_performance["tertiary_genre"] != "unknown"]

gold_genre_primary_performance["unified_genre"] = gold_genre_primary_performance[
    "primary_genre"]
gold_genre_secondary_performance["unified_genre"] = gold_genre_secondary_performance[
    "secondary_genre"]
gold_genre_tertiary_performance["unified_genre"] = gold_genre_tertiary_performance[
    "tertiary_genre"]

col_list = ["game_id", 
            "user_score", 
            "meta_score", 
            "current_price", 
            "stock_status", 
            "unified_genre"]

gold_genre_performance = pd.concat([gold_genre_primary_performance[col_list], 
                                    gold_genre_secondary_performance[col_list], 
                                    gold_genre_tertiary_performance[col_list]]
)
gold_genre_performance.head()

,game_id,user_score,meta_score,current_price,stock_status,unified_genre
0,the legend of zelda: ocarina of time,91,99,91.99,in_stock,action
1,super mario galaxy,91,97,91.99,out_of_stock,action
2,super mario galaxy 2,91,97,91.99,in_stock,action
3,metroid prime,89,97,89.99,out_of_stock,action
4,super mario odyssey,89,97,89.99,in_stock,action


## Grouping

Columns

número de juegos
precio medio
meta score medio
user score medio
porcentaje en stock
plataformas donde aparece
índice de atractivo del género

In [ ]:
gold_genre_performance = gold_genre_performance.groupby("unified_genre").agg(
     game_count=("game_id", "count"),
     average_user_score=("user_score", "mean"),
     average_meta_score=("meta_score", "mean"),
     average_price=("current_price", "mean"),
     in_stock_percentage=("stock_status", lambda x: (x == "in_stock").mean() * 100)
 )
gold_genre_performance["score_price"] = gold_genre_performance[
    "average_user_score"] / gold_genre_performance["average_price"]
gold_genre_performance.shape

               game_count  average_user_score  average_meta_score  \
unified_genre                                                       
action               1294           74.940495           77.194745   
adventure             268           71.600746           75.750000   
arcade                255           74.521569           77.431373   
board_game             50           68.900000           78.780000   
fighting               97           76.762887           77.814433   
horror                 35           77.885714           76.942857   
mmo                    48           71.062500           79.395833   
music                  73           74.917808           76.397260   
party                  33           72.818182           75.060606   
platformer            271           76.206642           77.944649   

               average_price  in_stock_percentage  score_price  
unified_genre                                                   
action             75.895719            5

In [5]:
gold_genre_performance.isna().sum()

game_count             0
average_user_score     0
average_meta_score     0
average_price          0
in_stock_percentage    0
score_price            0
dtype: int64

## Save Data

In [6]:
gold_genre_performance.to_parquet(DB_OUTPUT_GOLD / "gold_genre_performance.parquet")